# ISIC 2018 — 3-Stage Training Pipeline (Google Colab)

This notebook runs the full training pipeline on Google Colab with GPU:
1. **Stage 1 — YOLOv8**: Lesion detection (bounding box)
2. **Stage 2 — MedSAM**: Lesion segmentation (fine-tuned SAM)
3. **Stage 3 — Attribute Segmentation**: 5 dermoscopic attributes (SAM encoder + custom decoder)

## 0. Environment Setup

**How to use:**
1. Run the cells below to check GPU and install dependencies.
2. Upload the project zip (`belleai.zip`) when prompted — it will be extracted to `/content/belleai/`.
3. Alternatively, mount Google Drive if your data is already there.

In [1]:
!nvidia-smi


Wed Mar 11 10:24:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, zipfile, shutil
from pathlib import Path

PROJECT_ROOT = "/content/belleai"

# ── Option A: Upload a zip of the project ──────────────────────────────
# Before running this notebook, create a zip of the project folder:
#   cd /home/nghianq/Downloads && zip -r belleai.zip belleai/ \
#       -x "belleai/.venv/*" "belleai/__pycache__/*" "belleai/dataset/*" "belleai/outputs/*"
# This zips the source code + raw ISIC data (excluding venv/cache/prepared datasets).

if not os.path.exists(PROJECT_ROOT):
    from google.colab import files
    print("Upload belleai.zip (project zip with ISIC data + source code):")
    uploaded = files.upload()

    zip_name = list(uploaded.keys())[0]
    print(f"Extracting {zip_name}...")
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall("/content/")

    # Handle case where zip contains a top-level folder with different name
    if not os.path.exists(PROJECT_ROOT):
        extracted = [d for d in os.listdir("/content/") if os.path.isdir(f"/content/{d}") and d != "sample_data"]
        if extracted:
            os.rename(f"/content/{extracted[0]}", PROJECT_ROOT)

    os.remove(zip_name)
    print(f"Project extracted to {PROJECT_ROOT}")
else:
    print(f"Project already exists at {PROJECT_ROOT}")

# ── Option B: Mount Google Drive instead (uncomment if needed) ─────────
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = "/content/drive/MyDrive/belleai"

os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

# Verify data is present
for folder in ["ISIC2018_Task1-2_Training_Input", "ISIC2018_Task1_Training_GroundTruth",
               "ISIC2018_Task2_Training_GroundTruth_v3", "src"]:
    status = "✓" if os.path.isdir(folder) else "✗ MISSING"
    print(f"  {status}  {folder}")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/belleai'

In [ ]:
# Install dependencies
!pip install -q ultralytics segment-anything scikit-learn tqdm

# Download SAM ViT-B checkpoint (if not already present)
import os
ckpt_path = os.path.join(PROJECT_ROOT, "medsam_vit_b.pth")
if not os.path.exists(ckpt_path):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {ckpt_path}
    print("Downloaded SAM ViT-B checkpoint")
else:
    print("SAM checkpoint already exists")

In [ ]:
# Add src/ to Python path so imports like `from yolo.config import ...` work
import sys
src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(f"Python path includes: {src_path}")

---
## Stage 1 — YOLOv8 Lesion Detection

Prepare YOLO-format dataset from ISIC segmentation masks, then train YOLOv8 for bounding-box detection.

In [1]:
# 1.1 Prepare YOLO dataset
from yolo.prepare_data import get_image_ids, split_ids, prepare_yolo

image_ids = get_image_ids()
print(f"Found {len(image_ids)} images with segmentation masks")
train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train, {len(val_ids)} val")
prepare_yolo(train_ids, val_ids)

Found 2594 images with segmentation masks
Split: 2075 train, 519 val
YOLO dataset ready at /home/nghianq/Downloads/belleai/dataset/yolo  (2075 train, 519 val)


In [ ]:
# 1.2 Train YOLOv8
from yolo.train import train as train_yolo

model_yolo = train_yolo(epochs=100, batch=16, img_size=640)

In [ ]:
# 1.3 Validate YOLOv8
from yolo.train import validate as validate_yolo

validate_yolo()

---
## Stage 2 — MedSAM Lesion Segmentation

Prepare 1024×1024 npz files, then fine-tune MedSAM (SAM ViT-B) with bbox prompts for binary lesion segmentation.

In [ ]:
# 2.1 Prepare MedSAM dataset
from medsam.prepare_data import get_image_ids, split_ids, prepare_medsam

image_ids = get_image_ids()
train_ids, val_ids = split_ids(image_ids)
print(f"Split: {len(train_ids)} train, {len(val_ids)} val")
prepare_medsam(train_ids, val_ids)

In [ ]:
# 2.2 Train MedSAM
from medsam.train_segmentation import train as train_medsam

train_medsam(epochs=50, batch=4, lr=1e-4)

---
## Stage 3 — Attribute Segmentation

Prepare attribute dataset (5 dermoscopic attributes), then train the SAM encoder + custom decoder model.

In [ ]:
# 3.1 Prepare attribute dataset
from medsam.prepare_data import get_image_ids, split_ids, prepare_attributes

image_ids = get_image_ids()
train_ids, val_ids = split_ids(image_ids)
prepare_attributes(train_ids, val_ids)

In [ ]:
# 3.2 Train attribute segmentation
from medsam.train_attributes import train as train_attr

train_attr(epochs=80, batch=4, lr=1e-4)

---
## Inference Demo

Run the full 3-stage pipeline on a sample image.

In [ ]:
# Run inference on a sample image
from pathlib import Path
from inference import load_yolo, load_medsam, load_attr_model, predict
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

yolo_model = load_yolo()
medsam_model = load_medsam()
attr_model = load_attr_model()

# Pick a sample image
sample_img = sorted(
    list(Path(PROJECT_ROOT, "ISIC2018_Task1-2_Training_Input").glob("*.jpg"))
)[0]

bbox, lesion_mask, attr_masks = predict(str(sample_img), yolo_model, medsam_model, attr_model)

if lesion_mask is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    img = np.array(Image.open(sample_img))
    axes[0].imshow(img)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(lesion_mask, cmap="gray")
    axes[1].set_title("Lesion Segmentation")
    axes[1].axis("off")

    # Overlay attributes
    overlay = img.copy().astype(float) / 255.0
    colors = [(1,0,0), (0,1,0), (0,0,1), (1,1,0), (1,0,1)]
    for i, name in enumerate(attr_masks):
        m = attr_masks[name]
        for c in range(3):
            overlay[:,:,c] = np.where(m > 0, overlay[:,:,c] * 0.5 + colors[i][c] * 0.5, overlay[:,:,c])
    axes[2].imshow(overlay)
    axes[2].set_title("Attribute Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No lesion detected in sample image")

---
## Save Results

Download trained weights as a zip, or copy them to Google Drive.

In [ ]:
# Option A: Download outputs as zip
import zipfile
from google.colab import files

output_zip = "/content/trained_weights.zip"
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    outputs_dir = Path(PROJECT_ROOT, "outputs")
    if outputs_dir.exists():
        for f in outputs_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(PROJECT_ROOT))
                print(f"  Added: {f.relative_to(PROJECT_ROOT)}")
print(f"\nDownloading {output_zip}...")
files.download(output_zip)
